# Build Full 60 Connectivity Features

Notebook này dùng để tạo bộ feature đầy đủ hơn cho đồ án.

Nguồn hiện có:
- `Super_MultiDomain_Features_Role3`: đã có 30 feature
- `Cleaned_Epochs`: dữ liệu epoch `.fif` để tính thêm feature còn thiếu

Output mới:
- `Full_MultiDomain_Features_Role3`: folder mới chứa đủ 60 connectivity features theo paper

Cấu trúc sau khi chạy xong:
- `labels.npy`
- `subject_ids.npy`
- `all_feature_names.npy`
- 60 file feature dạng `band_metric.npy`, mỗi file shape `(13422, 19, 19)`


## 1. Import và cấu hình

Notebook dùng lại hàm `compute_feature_set(...)` trong `src/ftd_mlflow_pipeline/connectivity.py` để tính các metric còn thiếu từ `Cleaned_Epochs`.


In [1]:
from pathlib import Path
import os
import shutil
import sys
import time

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

import numpy as np
import pandas as pd

ROOT = Path('/home/dohaidang/DataMining_Project')
sys.path.insert(0, str(ROOT / 'src'))

from ftd_mlflow_pipeline.connectivity import FeatureSet, compute_feature_set


## 2. Đường dẫn và danh sách connectivity

Paper dùng 5 bands và 12 connectivity metrics. Super file hiện chỉ có 6 metrics, nên notebook sẽ tính thêm 6 metrics còn lại.


In [2]:
SUPER_DIR = ROOT / 'Super_MultiDomain_Features_Role3'
CLEANED_EPOCHS_DIR = ROOT / 'Cleaned_Epochs'
CACHE_DIR = ROOT / '.cache' / 'connectivity'
FULL_OUTPUT_DIR = ROOT / 'Full_MultiDomain_Features_Role3'

BANDS = ('delta', 'theta', 'alpha', 'beta', 'gamma')
PAPER_METRICS = (
    'cov', 'corr', 'xcov', 'xcorr', 'csd', 'coh',
    'mi', 'ecc', 'aecov', 'aecorr', 'plv', 'wplv',
)
SUPER_METRICS = ('cov', 'corr', 'plv', 'coh', 'csd', 'mi')
MISSING_METRICS = tuple(metric for metric in PAPER_METRICS if metric not in SUPER_METRICS)

OVERWRITE_EXISTING = False
SAVE_FULL_NPZ = False

print('Super folder:', SUPER_DIR)
print('Output folder:', FULL_OUTPUT_DIR)
print('Bands:', BANDS)
print('Paper metrics:', PAPER_METRICS)
print('Missing metrics:', MISSING_METRICS)
print('Expected total features:', len(BANDS) * len(PAPER_METRICS))
print('Expected missing features:', len(BANDS) * len(MISSING_METRICS))


Super folder: /home/dohaidang/DataMining_Project/Super_MultiDomain_Features_Role3
Output folder: /home/dohaidang/DataMining_Project/Full_MultiDomain_Features_Role3
Bands: ('delta', 'theta', 'alpha', 'beta', 'gamma')
Paper metrics: ('cov', 'corr', 'xcov', 'xcorr', 'csd', 'coh', 'mi', 'ecc', 'aecov', 'aecorr', 'plv', 'wplv')
Missing metrics: ('xcov', 'xcorr', 'ecc', 'aecov', 'aecorr', 'wplv')
Expected total features: 60
Expected missing features: 30


## 3. Kiểm tra Super folder

Cell này kiểm tra 30 feature đã có trong Super file.


In [3]:
def feature_name(band: str, metric: str) -> str:
    return f'{band}_{metric}'


def list_feature_files(folder: Path) -> list[str]:
    return sorted(
        path.stem
        for path in folder.glob('*.npy')
        if path.stem not in {'labels', 'subject_ids', 'all_feature_names', 'existing_feature_names', 'computed_feature_names'}
    )

super_features = list_feature_files(SUPER_DIR)
expected_super_features = [feature_name(band, metric) for band in BANDS for metric in SUPER_METRICS]
missing_from_super = sorted(set(expected_super_features) - set(super_features))
extra_in_super = sorted(set(super_features) - set(expected_super_features))

print('Super feature count:', len(super_features))
print('Missing expected Super features:', missing_from_super)
print('Extra Super features:', extra_in_super)

labels = np.load(SUPER_DIR / 'labels.npy', allow_pickle=True).astype(str)
subject_ids = np.load(SUPER_DIR / 'subject_ids.npy', allow_pickle=True).astype(str)

print('labels shape:', labels.shape)
print('subject_ids shape:', subject_ids.shape)
print('label distribution:', pd.Series(labels).value_counts().to_dict())
print('n_subjects:', pd.Series(subject_ids).nunique())


Super feature count: 30
Missing expected Super features: []
Extra Super features: []
labels shape: (13422,)
subject_ids shape: (13422,)
label distribution: {'A': 5666, 'C': 4603, 'F': 3153}
n_subjects: 88


## 4. Tạo output folder và copy 30 feature đã có

Các feature đã có sẽ được copy nguyên sang folder mới để tránh ghi đè Super folder gốc.


In [4]:
FULL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for file_name in ['labels.npy', 'subject_ids.npy']:
    src = SUPER_DIR / file_name
    dst = FULL_OUTPUT_DIR / file_name
    if OVERWRITE_EXISTING or not dst.exists():
        shutil.copy2(src, dst)

copied_features = []
for band in BANDS:
    for metric in SUPER_METRICS:
        name = feature_name(band, metric)
        src = SUPER_DIR / f'{name}.npy'
        dst = FULL_OUTPUT_DIR / f'{name}.npy'
        if not src.exists():
            raise FileNotFoundError(src)
        if OVERWRITE_EXISTING or not dst.exists():
            shutil.copy2(src, dst)
        copied_features.append(name)

print('Copied existing features:', len(copied_features))
print('Output folder now has:', len(list_feature_files(FULL_OUTPUT_DIR)), 'feature files')


Copied existing features: 30
Output folder now has: 49 feature files


## 5. Hàm kiểm tra alignment labels/subject_ids

Feature tính từ `Cleaned_Epochs` phải cùng thứ tự epoch với Super file. Cell này chuẩn hóa subject id để so sánh được `001` với `sub-001`.


In [5]:
def normalize_subject_id_array(values: np.ndarray, with_prefix: bool) -> np.ndarray:
    normalized = []
    for value in values.astype(str):
        clean = value.replace('sub-', '')
        clean = clean.zfill(3)
        normalized.append(f'sub-{clean}' if with_prefix else clean)
    return np.asarray(normalized)


def assert_alignment(feature_set: FeatureSet, labels_ref: np.ndarray, subjects_ref: np.ndarray):
    labels_feature = feature_set.group_codes.astype(str)
    subjects_feature = normalize_subject_id_array(feature_set.subject_ids.astype(str), with_prefix=False)
    subjects_reference = normalize_subject_id_array(subjects_ref.astype(str), with_prefix=False)

    if labels_feature.shape != labels_ref.shape:
        raise ValueError(f'Label shape mismatch: {labels_feature.shape} vs {labels_ref.shape}')
    if subjects_feature.shape != subjects_reference.shape:
        raise ValueError(f'Subject shape mismatch: {subjects_feature.shape} vs {subjects_reference.shape}')
    if not np.array_equal(labels_feature, labels_ref):
        mismatch = np.where(labels_feature != labels_ref)[0][:10]
        raise ValueError(f'Label order mismatch at indices: {mismatch}')
    if not np.array_equal(subjects_feature, subjects_reference):
        mismatch = np.where(subjects_feature != subjects_reference)[0][:10]
        raise ValueError(f'Subject order mismatch at indices: {mismatch}')

    return True


## 6. Tính 30 connectivity còn thiếu

Phần này sẽ tốn thời gian. Mỗi feature được cache trong `.cache/connectivity`, nên lần sau chạy lại sẽ nhanh hơn.

Các metrics được tính thêm:
- `xcov`
- `xcorr`
- `ecc`
- `aecov`
- `aecorr`
- `wplv`


In [6]:
computed_features = []
compute_log = []

for band in BANDS:
    for metric in MISSING_METRICS:
        name = feature_name(band, metric)
        output_path = FULL_OUTPUT_DIR / f'{name}.npy'
        if output_path.exists() and not OVERWRITE_EXISTING:
            print(f'Skip existing: {name}')
            computed_features.append(name)
            continue

        start = time.time()
        print(f'Computing {name} ...')
        feature_set = compute_feature_set(
            data_dir=CLEANED_EPOCHS_DIR,
            cache_dir=CACHE_DIR,
            band=band,
            metric=metric,
        )
        assert_alignment(feature_set, labels, subject_ids)
        np.save(output_path, feature_set.matrices.astype(np.float64))
        elapsed = time.time() - start

        computed_features.append(name)
        compute_log.append({
            'feature': name,
            'band': band,
            'metric': metric,
            'shape': feature_set.matrices.shape,
            'seconds': elapsed,
        })
        print(f'Saved {name}: {feature_set.matrices.shape}, {elapsed:.1f}s')

compute_log_df = pd.DataFrame(compute_log)
compute_log_df


Skip existing: delta_xcov
Skip existing: delta_xcorr
Skip existing: delta_ecc
Skip existing: delta_aecov
Skip existing: delta_aecorr
Skip existing: delta_wplv
Skip existing: theta_xcov
Skip existing: theta_xcorr
Skip existing: theta_ecc
Skip existing: theta_aecov
Skip existing: theta_aecorr
Skip existing: theta_wplv
Skip existing: alpha_xcov
Skip existing: alpha_xcorr
Skip existing: alpha_ecc
Skip existing: alpha_aecov
Skip existing: alpha_aecorr
Skip existing: alpha_wplv
Skip existing: beta_xcov
Computing beta_xcorr ...
Loading data for 119 events and 2500 original time points ...
Loading data for 158 events and 2500 original time points ...
Loading data for 61 events and 2500 original time points ...
Loading data for 135 events and 2500 original time points ...
Loading data for 135 events and 2500 original time points ...
Loading data for 127 events and 2500 original time points ...
Loading data for 153 events and 2500 original time points ...
Loading data for 159 events and 2500 ori

,feature,band,metric,shape,seconds
0,beta_xcorr,beta,xcorr,"(13422, 19, 19)",342.504330
1,beta_ecc,beta,ecc,"(13422, 19, 19)",533.610955
2,beta_aecov,beta,aecov,"(13422, 19, 19)",19.849147
3,beta_aecorr,beta,aecorr,"(13422, 19, 19)",20.438920
4,beta_wplv,beta,wplv,"(13422, 19, 19)",21.639887
5,gamma_xcov,gamma,xcov,"(13422, 19, 19)",342.304446
6,gamma_xcorr,gamma,xcorr,"(13422, 19, 19)",344.293092
7,gamma_ecc,gamma,ecc,"(13422, 19, 19)",559.580093
8,gamma_aecov,gamma,aecov,"(13422, 19, 19)",20.439202
9,gamma_aecorr,gamma,aecorr,"(13422, 19, 19)",20.783332


## 7. Lưu metadata danh sách feature

Cell này lưu danh sách feature đã có, feature mới tính và toàn bộ 60 feature.


In [7]:
all_feature_names = [feature_name(band, metric) for band in BANDS for metric in PAPER_METRICS]
existing_feature_names = [feature_name(band, metric) for band in BANDS for metric in SUPER_METRICS]
computed_feature_names = [feature_name(band, metric) for band in BANDS for metric in MISSING_METRICS]

np.save(FULL_OUTPUT_DIR / 'all_feature_names.npy', np.asarray(all_feature_names))
np.save(FULL_OUTPUT_DIR / 'existing_feature_names.npy', np.asarray(existing_feature_names))
np.save(FULL_OUTPUT_DIR / 'computed_feature_names.npy', np.asarray(computed_feature_names))

metadata = pd.DataFrame({
    'feature': all_feature_names,
    'band': [name.split('_', 1)[0] for name in all_feature_names],
    'metric': [name.split('_', 1)[1] for name in all_feature_names],
    'source': ['copied_from_super' if name in existing_feature_names else 'computed_from_cleaned_epochs' for name in all_feature_names],
})
metadata.to_csv(FULL_OUTPUT_DIR / 'feature_metadata.csv', index=False)
metadata


,feature,band,metric,source
0,delta_cov,delta,cov,copied_from_super
1,delta_corr,delta,corr,copied_from_super
2,delta_xcov,delta,xcov,computed_from_cleaned_epochs
3,delta_xcorr,delta,xcorr,computed_from_cleaned_epochs
4,delta_csd,delta,csd,copied_from_super
5,delta_coh,delta,coh,copied_from_super
6,delta_mi,delta,mi,copied_from_super
7,delta_ecc,delta,ecc,computed_from_cleaned_epochs
8,delta_aecov,delta,aecov,computed_from_cleaned_epochs
9,delta_aecorr,delta,aecorr,computed_from_cleaned_epochs


## 8. Kiểm tra output đủ 60 feature

Sau khi chạy xong, cell này xác nhận folder mới có đủ 60 file feature.


In [8]:
output_features = list_feature_files(FULL_OUTPUT_DIR)
expected_features = [feature_name(band, metric) for band in BANDS for metric in PAPER_METRICS]
missing_output = sorted(set(expected_features) - set(output_features))
extra_output = sorted(set(output_features) - set(expected_features))

print('Output feature count:', len(output_features))
print('Missing output:', missing_output)
print('Extra output:', extra_output)

sample_feature = np.load(FULL_OUTPUT_DIR / f'{expected_features[0]}.npy', allow_pickle=True)
print('Sample feature:', expected_features[0], sample_feature.shape, sample_feature.dtype)


Output feature count: 60
Missing output: []
Extra output: []
Sample feature: delta_cov (13422, 19, 19) float64


## 9. Tạo file NPZ tổng hợp tùy chọn

Mặc định `SAVE_FULL_NPZ = False` vì file tổng hợp có thể rất lớn. Nếu cần một file duy nhất, đổi `SAVE_FULL_NPZ = True` ở cell cấu hình.


In [9]:
if SAVE_FULL_NPZ:
    npz_path = ROOT / 'Full_MultiDomain_Features_Role3.npz'
    arrays = {
        name: np.load(FULL_OUTPUT_DIR / f'{name}.npy', allow_pickle=True)
        for name in all_feature_names
    }
    arrays['labels'] = np.load(FULL_OUTPUT_DIR / 'labels.npy', allow_pickle=True)
    arrays['subject_ids'] = np.load(FULL_OUTPUT_DIR / 'subject_ids.npy', allow_pickle=True)
    np.savez_compressed(npz_path, **arrays)
    print('Saved:', npz_path)
else:
    print('SAVE_FULL_NPZ = False, skip NPZ export')


SAVE_FULL_NPZ = False, skip NPZ export


## 10. Cách dùng output mới

Sau khi folder `Full_MultiDomain_Features_Role3` có đủ 60 feature, bạn có thể chỉnh các notebook training để trỏ sang folder này và dùng đủ 12 metrics:

```python
PRECOMPUTED_DIR = ROOT / 'Full_MultiDomain_Features_Role3'
SELECTED_BANDS = {'delta', 'theta', 'alpha', 'beta', 'gamma'}
SELECTED_METRICS = {'cov', 'corr', 'xcov', 'xcorr', 'csd', 'coh', 'mi', 'ecc', 'aecov', 'aecorr', 'plv', 'wplv'}
```
